# §1.2 接ベクトルと接空間 - パラメータ変化の方向を幾何学的に捉える

接ベクトルは「多様体上の点における方向」を表します。
統計的多様体では、パラメータの微小変化の方向に対応します。

## 既知概念との対応

| 情報幾何の概念 | 既知概念 |
|--------------|--------|
| 接ベクトル | パラメータの微小変化 $(d\mu, d\sigma)$ |
| 接空間 | その点で可能な全ての変化方向 |
| スコア関数 | 対数尤度の勾配（まさに接ベクトル的） |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle

plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
def gaussian_pdf(x, mu, sigma):
    """正規分布の確率密度関数"""
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

---
## 1. 接ベクトルの直感的理解

### 定義（直感的）

**接ベクトル** = 多様体上の点における「方向」

パラメータ空間の点 $p = (\mu_0, \sigma_0)$ における接ベクトルは、
パラメータの微小変化 $(d\mu, d\sigma)$ に対応する。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左図：パラメータ空間での接ベクトル
ax1 = axes[0]

# 基準点
mu0, sigma0 = 0, 1
ax1.plot(mu0, sigma0, 'ko', markersize=12, label=f'p = ({mu0}, {sigma0})')

# 接ベクトルの例
tangent_vectors = [
    ((1, 0), 'r', '∂/∂μ direction'),
    ((0, 0.5), 'b', '∂/∂σ direction'),
    ((0.7, 0.7), 'g', 'Mixed direction'),
    ((-0.5, 0.3), 'm', 'Another direction'),
]

for (dmu, dsigma), color, label in tangent_vectors:
    ax1.annotate('', xy=(mu0 + dmu, sigma0 + dsigma), xytext=(mu0, sigma0),
                 arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax1.text(mu0 + dmu * 1.1, sigma0 + dsigma * 1.1, label, fontsize=9, color=color)

ax1.set_xlabel('μ', fontsize=12)
ax1.set_ylabel('σ', fontsize=12)
ax1.set_title('Tangent vectors in parameter space\nRepresent "directions" at point p', fontsize=11)
ax1.set_xlim(-1.5, 2)
ax1.set_ylim(0, 2.5)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)

# 右図：分布空間での対応する変化
ax2 = axes[1]
x = np.linspace(-4, 4, 200)

# 基準分布
y0 = gaussian_pdf(x, mu0, sigma0)
ax2.plot(x, y0, 'k-', linewidth=2, label=f'p(x|{mu0},{sigma0}) base')

# 各方向への微小変化
epsilon = 0.3
changes = [
    ((mu0 + epsilon, sigma0), 'r', '--', 'μ+ε'),
    ((mu0, sigma0 + epsilon * 0.5), 'b', '--', 'σ+ε'),
    ((mu0 + epsilon * 0.7, sigma0 + epsilon * 0.7), 'g', '--', 'μ,σ+ε'),
]

for (mu, sigma), color, style, label in changes:
    y = gaussian_pdf(x, mu, sigma)
    ax2.plot(x, y, color=color, linestyle=style, linewidth=1.5, label=label)

ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('p(x)', fontsize=12)
ax2.set_title('Corresponding distribution changes\nTangent vector indicates direction of change', fontsize=11)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 学習ポイント

1. **接ベクトル** = パラメータの微小変化の「方向」
2. 点 $p = (\mu, \sigma)$ での接空間は、可能な全ての方向の集合
3. 正規分布の場合、接空間は2次元（$\partial/\partial\mu$ と $\partial/\partial\sigma$ が基底）
4. 接ベクトルは確率分布の変化方向を指定する

---
## 2. スコア関数と接ベクトルの関係

### スコア関数（Score Function）

**スコア関数** = 対数尤度のパラメータ微分

$$s_i(x; \theta) = \frac{\partial \log p(x|\theta)}{\partial \theta_i}$$

### 正規分布のスコア関数

$$\frac{\partial \log p(x|\mu,\sigma)}{\partial \mu} = \frac{x - \mu}{\sigma^2}$$

$$\frac{\partial \log p(x|\mu,\sigma)}{\partial \sigma} = -\frac{1}{\sigma} + \frac{(x-\mu)^2}{\sigma^3}$$

### 幾何学的意味
スコア関数は「接ベクトルを関数として表現」したもの。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

mu0, sigma0 = 0, 1
x = np.linspace(-4, 4, 200)

# 左上：確率密度関数
ax1 = axes[0, 0]
y = gaussian_pdf(x, mu0, sigma0)
ax1.plot(x, y, 'b-', linewidth=2)
ax1.fill_between(x, y, alpha=0.3)
ax1.set_xlabel('x')
ax1.set_ylabel('p(x)')
ax1.set_title(f'Probability density N({mu0}, {sigma0}²)')
ax1.grid(True, alpha=0.3)

# 右上：対数尤度
ax2 = axes[0, 1]
log_p = np.log(y + 1e-10)  # 数値安定性
ax2.plot(x, log_p, 'g-', linewidth=2)
ax2.set_xlabel('x')
ax2.set_ylabel('log p(x)')
ax2.set_title('Log-likelihood log p(x|μ,σ)')
ax2.grid(True, alpha=0.3)

# 左下：μに関するスコア関数
ax3 = axes[1, 0]
score_mu = (x - mu0) / sigma0**2
ax3.plot(x, score_mu, 'r-', linewidth=2)
ax3.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax3.set_xlabel('x')
ax3.set_ylabel('∂log p/∂μ')
ax3.set_title('Score function (μ direction)\n= (x-μ)/σ²')
ax3.grid(True, alpha=0.3)

# 右下：σに関するスコア関数
ax4 = axes[1, 1]
score_sigma = -1/sigma0 + (x - mu0)**2 / sigma0**3
ax4.plot(x, score_sigma, 'm-', linewidth=2)
ax4.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax4.set_xlabel('x')
ax4.set_ylabel('∂log p/∂σ')
ax4.set_title('Score function (σ direction)\n= -1/σ + (x-μ)²/σ³')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 学習ポイント

1. **スコア関数** = 対数尤度のパラメータ微分
2. これは「接ベクトルを関数として表現」したもの
3. 統計的多様体では、接ベクトルを関数（確率変数）として扱う
4. **Fisher情報行列 = スコア関数の共分散**
   $$I(\theta) = E\left[ \frac{\partial \log p}{\partial \theta} \cdot \frac{\partial \log p}{\partial \theta}^\top \right]$$
5. これがリーマン計量になる（次のノートブックで詳述）

---
## 3. 接空間の基底と座標表示

### 接空間 $T_pM$ は線形空間（ベクトル空間）

- 点 $p$ での接空間 $T_pM$ は線形空間
- 基底: $\{\partial/\partial\mu, \partial/\partial\sigma\}$
- 任意の接ベクトル: $v = a(\partial/\partial\mu) + b(\partial/\partial\sigma)$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左図：接空間の基底
ax1 = axes[0]
mu0, sigma0 = 0, 1

# 多様体上の点
ax1.plot(mu0, sigma0, 'ko', markersize=12)
ax1.text(mu0 - 0.2, sigma0 - 0.15, 'p', fontsize=14, fontweight='bold')

# 基底ベクトル
ax1.annotate('', xy=(mu0 + 1, sigma0), xytext=(mu0, sigma0),
             arrowprops=dict(arrowstyle='->', color='red', lw=3))
ax1.text(mu0 + 1.05, sigma0 + 0.05, '∂/∂μ', fontsize=12, color='red')

ax1.annotate('', xy=(mu0, sigma0 + 0.8), xytext=(mu0, sigma0),
             arrowprops=dict(arrowstyle='->', color='blue', lw=3))
ax1.text(mu0 + 0.05, sigma0 + 0.85, '∂/∂σ', fontsize=12, color='blue')

# 接空間を示す領域（概念的）
rect = Rectangle((mu0 - 0.8, sigma0 - 0.3), 2.2, 1.4,
                  fill=True, facecolor='yellow', alpha=0.2,
                  edgecolor='orange', linestyle='--')
ax1.add_patch(rect)
ax1.text(mu0 + 0.3, sigma0 + 0.9, 'Tangent space TₚM', fontsize=11, color='orange')

ax1.set_xlabel('μ', fontsize=12)
ax1.set_ylabel('σ', fontsize=12)
ax1.set_title('Basis vectors of tangent space\n{∂/∂μ, ∂/∂σ}', fontsize=11)
ax1.set_xlim(-1.5, 2)
ax1.set_ylim(0, 2.5)
ax1.grid(True, alpha=0.3)

# 右図：任意の接ベクトルの分解
ax2 = axes[1]
ax2.plot(mu0, sigma0, 'ko', markersize=12)

# 任意の接ベクトル v = 0.6(∂/∂μ) + 0.8(∂/∂σ)
a, b = 0.6, 0.8
ax2.annotate('', xy=(mu0 + a, sigma0 + b), xytext=(mu0, sigma0),
             arrowprops=dict(arrowstyle='->', color='green', lw=3))
ax2.text(mu0 + a + 0.05, sigma0 + b + 0.05, f'v = {a}(∂/∂μ) + {b}(∂/∂σ)',
         fontsize=11, color='green')

# 分解成分
ax2.annotate('', xy=(mu0 + a, sigma0), xytext=(mu0, sigma0),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5, linestyle='--'))
ax2.annotate('', xy=(mu0 + a, sigma0 + b), xytext=(mu0 + a, sigma0),
             arrowprops=dict(arrowstyle='->', color='blue', lw=1.5, linestyle='--'))

ax2.text(mu0 + a/2, sigma0 - 0.1, f'{a}(∂/∂μ)', fontsize=10, color='red')
ax2.text(mu0 + a + 0.05, sigma0 + b/2, f'{b}(∂/∂σ)', fontsize=10, color='blue')

ax2.set_xlabel('μ', fontsize=12)
ax2.set_ylabel('σ', fontsize=12)
ax2.set_title('Coordinate representation of tangent vector\nv = a(∂/∂μ) + b(∂/∂σ)', fontsize=11)
ax2.set_xlim(-1, 2)
ax2.set_ylim(0, 2.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 学習ポイント

1. 接空間 $T_pM$ は**線形空間**（ベクトル空間）
2. $n$次元多様体の接空間は$n$次元
3. 座標系 $(\theta^1, \theta^2, \ldots, \theta^n)$ に対して基底 $\{\partial/\partial\theta^i\}$
4. 任意の接ベクトルは基底の線形結合で表せる
5. 座標変換すると基底も変換される（共変性）

---
## 4. カルマンフィルタとの接点

### カルマンフィルタの状態更新を接ベクトルの視点から見る

カルマンフィルタでの状態更新:
- 予測分布 $N(\mu_{\text{pred}}, \sigma^2_{\text{pred}})$ から観測を得て
- 更新分布 $N(\mu_{\text{post}}, \sigma^2_{\text{post}})$ へ移動

この「移動」は**多様体上の曲線**として捉えられる。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# パラメータ
mu_pred, sigma_pred = 0, 2.0   # 事前分布
mu_obs = 1.5                    # 観測値
sigma_obs = 0.8                 # 観測ノイズ

# カルマンゲインと更新
K = sigma_pred**2 / (sigma_pred**2 + sigma_obs**2)
mu_post = mu_pred + K * (mu_obs - mu_pred)
sigma_post = np.sqrt((1 - K) * sigma_pred**2)

# 左図：パラメータ空間での更新
ax1 = axes[0]

# 事前、事後、観測の点
ax1.plot(mu_pred, sigma_pred, 'bo', markersize=12, label='Prior')
ax1.plot(mu_post, sigma_post, 'go', markersize=12, label='Posterior')
ax1.plot(mu_obs, sigma_obs, 'r^', markersize=12, label=f'Observation (μ_obs, σ_obs)')

# 更新の矢印（接ベクトル的）
ax1.annotate('', xy=(mu_post, sigma_post), xytext=(mu_pred, sigma_pred),
             arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax1.text((mu_pred + mu_post)/2 - 0.3, (sigma_pred + sigma_post)/2 + 0.1,
         'State update', fontsize=10, color='purple')

ax1.set_xlabel('μ', fontsize=12)
ax1.set_ylabel('σ', fontsize=12)
ax1.set_title('Kalman filter state update\nMovement on manifold', fontsize=11)
ax1.legend()
ax1.set_xlim(-1, 3)
ax1.set_ylim(0, 3)
ax1.grid(True, alpha=0.3)

# 右図：対応する分布の変化
ax2 = axes[1]
x = np.linspace(-5, 5, 200)

y_pred = gaussian_pdf(x, mu_pred, sigma_pred)
y_post = gaussian_pdf(x, mu_post, sigma_post)
y_obs = gaussian_pdf(x, mu_obs, sigma_obs)

ax2.plot(x, y_pred, 'b-', linewidth=2, label=f'Prior N({mu_pred:.1f}, {sigma_pred:.1f}²)')
ax2.plot(x, y_post, 'g-', linewidth=2, label=f'Posterior N({mu_post:.2f}, {sigma_post:.2f}²)')
ax2.plot(x, y_obs, 'r--', linewidth=2, label=f'Likelihood N({mu_obs:.1f}, {sigma_obs:.1f}²)')
ax2.axvline(x=mu_obs, color='r', linestyle=':', alpha=0.5, label='Observation')

ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('p(x)', fontsize=12)
ax2.set_title('Corresponding distribution changes', fontsize=11)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"""
【Kalman filter update】
- Prior: (μ, σ) = ({mu_pred:.1f}, {sigma_pred:.1f})
- Posterior: (μ, σ) = ({mu_post:.2f}, {sigma_post:.2f})
- Kalman gain K = {K:.3f}
""")

### 📝 カルマンフィルタと情報幾何の接点

1. カルマンフィルタの状態更新は**多様体上の「移動」**
2. 更新ベクトル $(\Delta\mu, \Delta\sigma)$ は接ベクトル的
3. 情報幾何では、この移動を「測地線」や「射影」で解釈できる
4. 自然勾配法もこの視点から理解できる（第4章で詳述）

---
## 5. 確認問題

### Q1. 接ベクトルの成分
点 $p = (\mu, \sigma) = (1, 2)$ での接ベクトル $v = 3(\partial/\partial\mu) - (\partial/\partial\sigma)$ は、どの方向を指すか図示せよ。

### Q2. スコア関数
ベルヌーイ分布 $\text{Ber}(p)$ のスコア関数 $\partial \log P(x|p)/\partial p$ を計算せよ。

### Q3. カルマンフィルタとの対応
カルマンゲインが大きいとき、パラメータ空間での移動ベクトルはどうなるか説明せよ。

In [ ]:
# Q1の解答
fig, ax = plt.subplots(figsize=(8, 6))

mu_p, sigma_p = 1, 2
ax.plot(mu_p, sigma_p, 'ko', markersize=12, label='p = (1, 2)')

# v = 3(∂/∂μ) - (∂/∂σ) → 成分は (3, -1)
v_mu, v_sigma = 3, -1
scale = 0.3  # 可視化用のスケール
ax.annotate('', xy=(mu_p + v_mu*scale, sigma_p + v_sigma*scale), xytext=(mu_p, sigma_p),
             arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(mu_p + v_mu*scale + 0.1, sigma_p + v_sigma*scale, 'v = (3, -1)', fontsize=11, color='red')

ax.set_xlabel('μ')
ax.set_ylabel('σ')
ax.set_title('Q1: Tangent vector v = 3(∂/∂μ) - (∂/∂σ)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 3)
ax.set_ylim(0, 3)
plt.show()

print("Q1: μ方向に+3、σ方向に-1 → 右下方向を指す")

In [ ]:
# Q2の解答
print("""
Q2 解答:
ベルヌーイ分布: P(x|p) = p^x (1-p)^{1-x}  (x ∈ {0, 1})

対数尤度:
log P(x|p) = x log p + (1-x) log(1-p)

スコア関数:
∂ log P(x|p)/∂p = x/p - (1-x)/(1-p)
                = x/p - (1-x)/(1-p)
                = (x(1-p) - (1-x)p) / (p(1-p))
                = (x - p) / (p(1-p))

確認:
- E[∂ log P/∂p] = E[x - p]/(p(1-p)) = 0 ✓ (スコア関数の期待値は0)
- Fisher情報: I(p) = E[(∂ log P/∂p)²] = Var(x)/(p(1-p))² = 1/(p(1-p))
""")

In [ ]:
# Q3の解答
print("""
Q3 解答:
カルマンゲイン K が大きいとき:

1. K が大きい条件:
   K = σ_pred² / (σ_pred² + σ_obs²)
   → σ_obs が小さい（観測精度が高い）とき K ≈ 1

2. 更新量:
   Δμ = K(μ_obs - μ_pred) → K大 ⇒ 観測値に大きく引っ張られる
   σ_post = √((1-K)σ_pred²) → K大 ⇒ σが大きく減少

3. パラメータ空間での移動:
   - μ方向: 観測値に向かって大きく移動
   - σ方向: 大きく減少（不確実性が減る）
   - 結果: 観測点に近づく方向への大きな移動

4. 情報幾何的解釈:
   - 高精度観測 = 尤度関数が鋭い
   - Fisher的に「情報量が多い」
   - 多様体上で「遠く」まで移動できる
""")

---
## まとめ

| 概念 | 説明 |
|-----|------|
| 接ベクトル | 多様体上の点での「方向」、パラメータの微小変化に対応 |
| 接空間 $T_pM$ | 点 $p$ での全ての接ベクトルの集合（線形空間）|
| 基底 | $\{\partial/\partial\theta^i\}$、座標系に依存 |
| スコア関数 | 接ベクトルの「関数表現」|

---
**次のノートブック**: `sec03_metric_tensor.ipynb` - リーマン計量（Fisher情報行列）